# EcoData · S6 — Ingesta de eventos empresariales en tiempo real (dimensión clima)
**Alahin Reyme Ticona Veliz** · Proyecto Sello EcoData (dimensión **clima / temperatura**, Juliaca) · Big Data, UPeU

La guía de S6 usa *orden → pago*. Aquí se aplica el **mismo patrón** al negocio de la dimensión clima: avisar de **heladas** en Juliaca.

| Guía S6 | EcoData · clima | Topic | Key |
|---|---|---|---|
| `ec-orden-ms` publica `orden.creada` | **`ec-frio-ms`** publica `clima.frio.detectado` (temperatura bajo 2 °C) | `ecodata-clima-episodios` | `episodioId` |
| `ec-pago-ms` consume y publica `pago.aprobado` / `pago.rechazado` | **`ec-helada-ms`** consume y publica `helada.emitida` / `helada.descartada` | `ecodata-clima-alertas` | `episodioId` |
| evento propio (`orden.cancelada`) | **`helada.cancelada`** (la temperatura volvió sobre el umbral) | `ecodata-clima-alertas` | `episodioId` |

Los datos de entrada son **reales** (Open-Meteo, temperatura a 2 m de Juliaca), no inventados. Los topics son **nuevos** (`ecodata-clima-*`): no tocan `atmos-clima` ni `atmos-negocio`, así que no alteran Grafana ni las alertas de Prometheus de S12.

> Requisitos: Kafka levantado (Docker) y este cuaderno ejecutándose dentro del contenedor de Jupyter. Ejecuta las secciones **en orden** y no repitas las de publicación (duplican eventos).

**Prueba por consola (3.2 de la guía)** — hazla antes, en CMD, y captura la pantalla completa (con reloj y perfil):
```
docker exec -it lambda26-kafka /opt/kafka/bin/kafka-topics.sh --create --topic ecodata-clima-prueba-consola --bootstrap-server kafka:9092 --partitions 1 --replication-factor 1
docker exec -it lambda26-kafka /opt/kafka/bin/kafka-console-consumer.sh --topic ecodata-clima-prueba-consola --bootstrap-server kafka:9092 --from-beginning
```
y en **otra** ventana CMD:
```
docker exec -it lambda26-kafka /opt/kafka/bin/kafka-console-producer.sh --topic ecodata-clima-prueba-consola --bootstrap-server kafka:9092
```
Escribe `hola clima` y comprueba que aparece en el consumer.

## 0. Preparación

In [1]:
import sys, json, time
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import clima_comun as clima
import openmeteo_clima_a_kafka as pub_clima

from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "kafka:9092"
TOPIC_EPIS, TOPIC_ALERT = "ecodata-clima-episodios", "ecodata-clima-alertas"
UMBRAL_FRIO = 2.0       # °C: bajo este valor se detecta un episodio de frío
NIVEL_HELADA = 0.0      # °C: bajo 0 °C a 2 m es helada (mismo umbral del job de producción s12_streaming_clima.py)
SENSOR = clima.SENSOR_ID
HISTORICO = "/tmp/s12/clima_juliaca_2023.json"
print(f"Umbral de episodio: < {UMBRAL_FRIO} °C · umbral de helada: < {NIVEL_HELADA} °C · sensor: {SENSOR}")

def log(**kv):   # formato de la guía: cada línea declara service, component, topic y status
    print(" ".join(f"{k}={v}" for k, v in kv.items()))

Umbral de episodio: < 2.0 °C · umbral de helada: < 0.0 °C · sensor: openmeteo-clima-juliaca


## 1. Crear los topics de forma explícita (3.1–3.2)
La guía deja `KAFKA_AUTO_CREATE_TOPICS_ENABLE=false`; en el clúster del proyecto un productor puede crear un topic solo (con 1 partición). Por eso se crean aquí **con 3 particiones**: así la *key* (`episodioId`) reparte los eventos y se ve el particionado en Kafka UI.

In [2]:
adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
for t in (TOPIC_EPIS, TOPIC_ALERT):
    try:
        adm.create_topics([NewTopic(t, num_partitions=3, replication_factor=1)])
        print("creado:", t)
    except TopicAlreadyExistsError:
        print("ya existía:", t)
adm.close()
c = KafkaConsumer(bootstrap_servers=BOOTSTRAP)
print(sorted(t for t in c.topics() if t.startswith("ecodata-clima")))
c.close()

creado: ecodata-clima-episodios
creado: ecodata-clima-alertas
['ecodata-clima-alertas', 'ecodata-clima-episodios']


## 2. Contratos de los eventos (3.9 y 4.1)
**`clima.frio.detectado`** — topic `ecodata-clima-episodios`, key `episodioId`, productor `ec-frio-ms`, consumidor `ec-helada-ms`
```json
{"tipoEvento":"clima.frio.detectado","episodioId":"openmeteo-clima-juliaca-1688180400000",
 "sensorId":"openmeteo-clima-juliaca","temp":-2.4,"hum":58.0,"umbral":2.0,"origen":"openmeteo-replay","timestamp":1688180400000}
```
**`helada.emitida` / `helada.descartada`** — topic `ecodata-clima-alertas`, key `episodioId`, productor `ec-helada-ms`. Regla: temperatura < 0 °C → *emitida*; entre 0 y 2 °C → *descartada* (frío sin helada, solo informativo; equivale a `pago.rechazado` de la guía).
```json
{"tipoEvento":"helada.emitida","episodioId":"…","sensorId":"…","temp":-2.4,"nivel":"ALERTA","motivo":"temp<0","origen":"ec-helada-ms","timestamp":1688180400000}
```
**`helada.cancelada` (evento propio, 4.1)** — topic `ecodata-clima-alertas`, key `episodioId`, productor `ec-helada-ms`, consumidor `ec-clima-notificacion`. Se publica cuando, una hora después, la temperatura vuelve a ser ≥ 2 °C.
```json
{"tipoEvento":"helada.cancelada","episodioId":"…","sensorId":"…","temp":3.1,"nivel":"CANCELADA","motivo":"temp sobre umbral en la hora siguiente","origen":"ec-helada-ms","timestamp":1688184000000}
```
| Campo | Tipo | Unidad / regla |
|---|---|---|
| `tipoEvento` | string | uno de los cuatro tipos |
| `episodioId` | string | `sensorId` + hora del dato (epoch ms): idempotente |
| `temp` | número | °C a 2 m, entre −60 y 60 |
| `hum` | número | % (0–100), solo en el evento de entrada |
| `umbral` | número | °C |
| `timestamp` | entero | hora del dato, epoch ms UTC |

**Por qué `episodioId` como key:** todos los eventos de un mismo episodio (detectado → emitida → cancelada) caen en la misma partición y se leen en orden entre sí, igual que `ordenId` en la guía.

## 3. Datos reales para los eventos
Se toman los episodios reales de la API de Open-Meteo (horas con temperatura bajo 2 °C). La API reciente trae solo ~50 h y en esta época casi no hay heladas, así que si hay pocas se completa con el **histórico real de Juliaca** del proyecto (`origen = openmeteo-replay`).

In [3]:
def desde_archivo(path=HISTORICO):
    return clima.leer_json(path)                       # [(ts_ms, temp, hum, viento)]

try:
    serie_api = pub_clima.desde_api()
except Exception as ex:
    print("API no disponible:", type(ex).__name__); serie_api = []
fuente = {t: x for t, x, h, v in serie_api}
pool = [(t, x, h, "openmeteo-api") for t, x, h, v in serie_api if x < UMBRAL_FRIO]
if len(pool) < 40 or sum(1 for p in pool if p[1] < NIVEL_HELADA) < 6:
    print(f"La API trajo {len(serie_api)} h y {len(pool)} episodios; completo con el histórico real de Juliaca")
    hist = desde_archivo()
    fuente = {t: x for t, x, h, v in hist}
    pool = [(t, x, h, "openmeteo-replay") for t, x, h, v in hist if x < UMBRAL_FRIO]

# Lotes: garantizan que haya heladas (emitida) y frío leve (descartada)
heladas = [p for p in pool if p[1] < NIVEL_HELADA]
frescas = [p for p in pool if NIVEL_HELADA <= p[1] < UMBRAL_FRIO]
LOTE1 = heladas[:5] + frescas[:5]
# Heladas cuya hora siguiente vuelve a ≥ umbral: son las que se pueden cancelar (sección 6)
CANCELABLES = [p for p in heladas if p not in LOTE1 and fuente.get(p[0] + 3600_000, -99) >= UMBRAL_FRIO][:4]
RAFAGA = [p for p in pool if p not in LOTE1 and p not in CANCELABLES][:30]
print(f"{len(pool)} episodios reales ({len(heladas)} heladas) · lote 1: {len(LOTE1)} · cancelables: {len(CANCELABLES)} · ráfaga: {len(RAFAGA)}")
print("temperatura mínima disponible:", round(min(p[1] for p in pool), 1), "°C el", pd.to_datetime(min(pool, key=lambda p: p[1])[0], unit="ms"), "UTC")

La API trajo 52 h y 0 episodios; completo con el histórico real de Juliaca
692 episodios reales (296 heladas) · lote 1: 10 · cancelables: 4 · ráfaga: 30
temperatura mínima disponible: -4.2 °C el 2023-06-10 11:00:00 UTC


## 4. Servicio productor `ec-frio-ms` (3.5–3.6)
Publica `clima.frio.detectado` y registra el resultado real del broker (partición y offset), como el `whenComplete` de la guía.

In [4]:
prod = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all",
                     key_serializer=str.encode, value_serializer=lambda v: v if isinstance(v, bytes) else json.dumps(v).encode())

def publicar(servicio, topic, ev, key):
    try:
        md_ = prod.send(topic, key=key, value=ev).get(timeout=10)
        log(service=servicio, component="producer", topic=md_.topic, partition=md_.partition, offset=md_.offset,
            eventType=ev["tipoEvento"], episodioId=ev["episodioId"], timestamp=ev["timestamp"], status="published")
    except Exception as e:
        log(service=servicio, component="producer", topic=topic, eventType=ev.get("tipoEvento"), status="error", error=repr(e)[:80])

def episodio(ts, temp, hum, origen):
    return {"tipoEvento": "clima.frio.detectado", "episodioId": f"{SENSOR}-{ts}", "sensorId": SENSOR,
            "temp": round(float(temp), 2), "hum": round(float(hum), 1), "umbral": UMBRAL_FRIO, "origen": origen, "timestamp": ts}

for ts, x, h, org in LOTE1:                      # ¡ejecutar UNA sola vez!
    ev = episodio(ts, x, h, org); publicar("ec-frio-ms", TOPIC_EPIS, ev, ev["episodioId"])

service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=0 offset=0 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1673690400000 timestamp=1673690400000 status=published
service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=0 offset=1 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1680778800000 timestamp=1680778800000 status=published
service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=1 offset=0 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1682330400000 timestamp=1682330400000 status=published
service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=1 offset=1 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1682334000000 timestamp=1682334000000 status=published
service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=1 offset=2 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-16837

## 5. Servicio `ec-helada-ms`: consumidor y productor (3.7–3.8)
Consume `ecodata-clima-episodios` con su propio *consumer group* (`ec-helada-ms-group`), decide y publica el resultado en `ecodata-clima-alertas`. **Valida cada mensaje**: uno malformado se marca `invalid` y el servicio sigue (error frecuente de 2.2 y 3.4).

In [5]:
REQ = {"tipoEvento": str, "episodioId": str, "sensorId": str, "temp": (int, float), "hum": (int, float),
       "umbral": (int, float), "origen": str, "timestamp": int}

def validar_episodio(ev):
    if not isinstance(ev, dict): return "el payload no es un objeto JSON"
    for k, tipo in REQ.items():
        if k not in ev: return f"falta el campo {k}"
        if isinstance(ev[k], bool) or not isinstance(ev[k], tipo): return f"tipo inválido en {k}"
    if ev["tipoEvento"] != "clima.frio.detectado": return "tipoEvento inesperado"
    if not -60 <= ev["temp"] <= 60: return "temperatura fuera de rango físico"
    return None

def servicio_helada(espera_s=8):
    cons = KafkaConsumer(TOPIC_EPIS, bootstrap_servers=BOOTSTRAP, group_id="ec-helada-ms-group",
                         auto_offset_reset="earliest", enable_auto_commit=True,
                         consumer_timeout_ms=int(espera_s * 1000))
    n = {"emitida": 0, "descartada": 0, "invalid": 0}
    for m in cons:
        try:
            ev = json.loads(m.value.decode("utf-8"))
            err = validar_episodio(ev)
        except Exception as e:
            ev, err = None, f"JSON ilegible ({type(e).__name__})"
        if err:
            n["invalid"] += 1
            log(service="ec-helada-ms", component="consumer", topic=m.topic, partition=m.partition, offset=m.offset,
                status="invalid", error=f'"{err}"', rawPayload=m.value[:70])
            continue
        log(service="ec-helada-ms", component="consumer", topic=m.topic, partition=m.partition, offset=m.offset,
            eventType=ev["tipoEvento"], episodioId=ev["episodioId"], status="received")
        emitida = ev["temp"] < NIVEL_HELADA
        out = {"tipoEvento": "helada.emitida" if emitida else "helada.descartada", "episodioId": ev["episodioId"],
               "sensorId": ev["sensorId"], "temp": ev["temp"], "nivel": "ALERTA" if emitida else "INFORMATIVO",
               "motivo": "temp<0" if emitida else "0<=temp<2 (frío sin helada)", "origen": "ec-helada-ms", "timestamp": ev["timestamp"]}
        publicar("ec-helada-ms", TOPIC_ALERT, out, ev["episodioId"])
        n["emitida" if emitida else "descartada"] += 1
    cons.close()
    return n

print(servicio_helada())

service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=0 offset=0 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1673690400000 status=received
service=ec-helada-ms component=producer topic=ecodata-clima-alertas partition=0 offset=0 eventType=helada.emitida episodioId=openmeteo-clima-juliaca-1673690400000 timestamp=1673690400000 status=published
service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=0 offset=1 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1680778800000 status=received
service=ec-helada-ms component=producer topic=ecodata-clima-alertas partition=0 offset=1 eventType=helada.emitida episodioId=openmeteo-clima-juliaca-1680778800000 timestamp=1680778800000 status=published
service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=0 offset=2 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1673344800000 status=received
service=ec-helada-ms component=

## 6. Evento propio nuevo: `helada.cancelada` (4.1, puntos 1–2)
Para cada helada emitida, si **la hora siguiente** la temperatura volvió a ser ≥ 2 °C, `ec-helada-ms` publica su cancelación. Como la hora siguiente de una helada no siempre sube tanto, el cuaderno ya separó en `CANCELABLES` heladas reales con esa propiedad: se publican, se procesan y se cancelan.

In [6]:
def cancelar_alertas(lote):
    c = 0
    for ts, x, h, org in lote:
        if x >= NIVEL_HELADA: continue                      # solo las que se emitieron
        sig = fuente.get(ts + 3600_000)
        if sig is not None and sig >= UMBRAL_FRIO:
            ev = {"tipoEvento": "helada.cancelada", "episodioId": f"{SENSOR}-{ts}", "sensorId": SENSOR,
                  "temp": round(float(sig), 2), "nivel": "CANCELADA", "motivo": "temp sobre umbral en la hora siguiente",
                  "origen": "ec-helada-ms", "timestamp": ts + 3600_000}
            publicar("ec-helada-ms", TOPIC_ALERT, ev, ev["episodioId"]); c += 1
    return c

# 1) publicar episodios cancelables  2) que el servicio los procese  3) cancelar        ¡UNA sola vez!
for ts, x, h, org in CANCELABLES:
    ev = episodio(ts, x, h, org); publicar("ec-frio-ms", TOPIC_EPIS, ev, ev["episodioId"])
print(servicio_helada())
print("cancelaciones publicadas:", cancelar_alertas(LOTE1 + CANCELABLES))

service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=1 offset=6 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1684238400000 timestamp=1684238400000 status=published
service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=1 offset=7 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1684756800000 timestamp=1684756800000 status=published
service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=1 offset=8 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1685448000000 timestamp=1685448000000 status=published
service=ec-frio-ms component=producer topic=ecodata-clima-episodios partition=2 offset=0 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1685534400000 timestamp=1685534400000 status=published
service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=1 offset=6 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-168

### 6.1 Segundo consumidor sobre el mismo topic
`ec-clima-notificacion` lee `ecodata-clima-alertas` con **otro consumer group**: recibe una copia completa de los eventos sin quitárselos a nadie (responde «¿por qué Kafka no borra un evento al leerlo?»). El timeout es largo porque el primer `poll` de un grupo nuevo tarda en unirse; si sale vacío, vuelve a correr la celda.

In [7]:
cons = KafkaConsumer(TOPIC_ALERT, bootstrap_servers=BOOTSTRAP, group_id="ec-clima-notificacion-group",
                     auto_offset_reset="earliest", consumer_timeout_ms=15000)
filas = []
for m in cons:
    ev = json.loads(m.value); filas.append((m.partition, m.offset, ev["tipoEvento"], ev["episodioId"][-13:], ev["temp"]))
cons.close()
df = pd.DataFrame(filas, columns=["partition", "offset", "tipoEvento", "hora_ms", "temp"])
print(df.tipoEvento.value_counts().to_string()); df.sort_values(["partition", "offset"]).head(12)

tipoEvento
helada.emitida       9
helada.cancelada     5
helada.descartada    5


,partition,offset,tipoEvento,hora_ms,temp
2,0,0,helada.emitida,1673690400000,-0.3
3,0,1,helada.emitida,1680778800000,-0.1
4,0,2,helada.descartada,1673344800000,1.1
5,0,3,helada.descartada,1673686800000,0.9
6,1,0,helada.emitida,1682330400000,-0.2
7,1,1,helada.emitida,1682334000000,-0.5
8,1,2,helada.emitida,1683716400000,-0.6
9,1,3,helada.descartada,1673341200000,1.8
10,1,4,helada.descartada,1673348400000,1.4
11,1,5,helada.descartada,1673694000000,1.2


## 7. Mensaje que incumple el contrato (4.1, punto 3)
Se publican a propósito dos mensajes malos en `ecodata-clima-episodios`: texto plano (como el `hola` de la consola) y un JSON **sin `episodioId`**. El consumidor no debe caerse.

In [8]:
prod.send(TOPIC_EPIS, key="malo-1", value=b"hola clima (texto plano)")
prod.send(TOPIC_EPIS, key="malo-2",
          value=json.dumps({"tipoEvento": "clima.frio.detectado", "sensorId": SENSOR, "temp": -3.0, "hum": 50.0,
                            "umbral": UMBRAL_FRIO, "origen": "prueba", "timestamp": 1}).encode())
prod.flush()
print(servicio_helada())      # debe mostrar status=invalid ×2 y seguir funcionando

service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=1 offset=9 status=invalid error="JSON ilegible (JSONDecodeError)" rawPayload=b'hola clima (texto plano)'
service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=1 offset=10 status=invalid error="falta el campo episodioId" rawPayload=b'{"tipoEvento": "clima.frio.detectado", "sensorId": "openmeteo-clima-ju'
{'emitida': 0, 'descartada': 0, 'invalid': 2}


## 8. Lag del *consumer group* antes y después de una ráfaga (4.1, punto 4)
*Lag* = mensajes publicados que el grupo todavía no leyó. Se mide con la API y se compara con Kafka UI (Topics → `ecodata-clima-episodios` → Consumers) y con la consola:
```
docker exec -it lambda26-kafka /opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server kafka:9092 --describe --group ec-helada-ms-group
```

In [9]:
def lag(group, topic):
    # consumidor con el mismo group_id, sin suscribirse: solo consulta el offset confirmado (no altera el grupo)
    c = KafkaConsumer(bootstrap_servers=BOOTSTRAP, group_id=group, enable_auto_commit=False)
    tps = [TopicPartition(topic, p) for p in sorted(c.partitions_for_topic(topic))]
    fin = c.end_offsets(tps)
    filas = []
    for t in tps:
        cur = c.committed(t) or 0
        filas.append((t.partition, cur, fin[t], fin[t] - cur))
    c.close()
    return pd.DataFrame(filas, columns=["partición", "offset_leído", "offset_final", "lag"])
print("ANTES de la ráfaga"); a = lag("ec-helada-ms-group", TOPIC_EPIS); display(a); print("lag total:", a.lag.sum())

ANTES de la ráfaga


,partición,offset_leído,offset_final,lag
0,0,4,4,0
1,1,11,11,0
2,2,1,1,0


lag total: 0


In [10]:
for ts, x, h, org in RAFAGA:                           # ¡ejecutar UNA sola vez!
    ev = episodio(ts, x, h, org); prod.send(TOPIC_EPIS, key=ev["episodioId"], value=ev)
prod.flush()
print(f"ráfaga publicada: {len(RAFAGA)} eventos")
print("DESPUÉS de publicar, SIN consumir"); b = lag("ec-helada-ms-group", TOPIC_EPIS); display(b); print("lag total:", b.lag.sum())

ráfaga publicada: 30 eventos
DESPUÉS de publicar, SIN consumir


,partición,offset_leído,offset_final,lag
0,0,4,17,13
1,1,11,20,9
2,2,1,9,8


lag total: 30


In [11]:
print(servicio_helada())
print("DESPUÉS de consumir"); d = lag("ec-helada-ms-group", TOPIC_EPIS); display(d); print("lag total:", d.lag.sum())

service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=0 offset=4 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1674381600000 status=received
service=ec-helada-ms component=producer topic=ecodata-clima-alertas partition=0 offset=4 eventType=helada.descartada episodioId=openmeteo-clima-juliaca-1674381600000 timestamp=1674381600000 status=published
service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=0 offset=5 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1680346800000 status=received
service=ec-helada-ms component=producer topic=ecodata-clima-alertas partition=0 offset=5 eventType=helada.descartada episodioId=openmeteo-clima-juliaca-1680346800000 timestamp=1680346800000 status=published
service=ec-helada-ms component=consumer topic=ecodata-clima-episodios partition=0 offset=6 eventType=clima.frio.detectado episodioId=openmeteo-clima-juliaca-1680782400000 status=received
service=ec-helada-ms comp

,partición,offset_leído,offset_final,lag
0,0,17,17,0
1,1,20,20,0
2,2,9,9,0


lag total: 0


### Qué capturar (con **reloj del sistema y tu perfil visibles, pantalla completa**)
1. CMD con la prueba por consola (topic `ecodata-clima-prueba-consola`).
2. Kafka UI → Topics: `ecodata-clima-episodios` y `ecodata-clima-alertas` con 3 particiones y mensajes visibles.
3. Kafka UI → un mensaje abierto de cada topic.
4. Las salidas de las secciones 4, 5 y 6 (logs `component=producer` / `consumer`).
5. La sección 7 con `status=invalid` y el consumidor siguiendo.
6. Las tres tablas de lag de la sección 8 (antes, sin consumir, después).

Si algo falla, **no repitas** las secciones de publicación: avisa con la captura.

In [12]:
prod.close()